## Data Cleaning Framework

In [1]:
import pandas as pd
import numpy as np

In [2]:
zomato = pd.read_csv("zomato.csv")

In [3]:
zomato.head()

,Restaurant_Name,Dining_Rating,Delivery_Rating,Dining Votes,Delivery_Votes,Cuisine,Place_Name,City,Item_Name,Best_Seller,Votes,Prices
0,Doner King,3.9,4.2,39,0,Fast Food,Malakpet,Hyderabad,Platter Kebab Combo,BESTSELLER,84,249.0
1,Doner King,3.9,4.2,39,0,Fast Food,Malakpet,Hyderabad,Chicken Rumali Shawarma,BESTSELLER,45,129.0
2,Doner King,3.9,4.2,39,0,Fast Food,Malakpet,Hyderabad,Chicken Tandoori Salad,NaN,39,189.0
3,Doner King,3.9,4.2,39,0,Fast Food,Malakpet,Hyderabad,Chicken BBQ Salad,BESTSELLER,43,189.0
4,Doner King,3.9,4.2,39,0,Fast Food,Malakpet,Hyderabad,Special Doner Wrap Combo,MUST TRY,31,205.0


### A. Structural Cleaning

#### A1. Rename Columns to a Standard Format

In [4]:
zomato.columns

Index(['Restaurant_Name', 'Dining_Rating', 'Delivery_Rating', 'Dining Votes',
       'Delivery_Votes', 'Cuisine ', 'Place_Name', 'City', 'Item_Name',
       'Best_Seller', 'Votes', 'Prices'],
      dtype='object')

In [5]:
zomato.rename(columns={'Restaurant_Name': 'restaurant_name','Dining_Rating':'dining_rating','Delivery_Rating':'delivery_rating','Dining Votes':'dining_votes','Delivery_Votes':'delivery_votes','Cuisine ':'cuisine','Place_Name':'place_name','City':'city','Item_Name':'item_name','Best_Seller':'best_seller','Votes':'item_votes','Prices':'item_price'},inplace=True)

In [6]:
zomato.columns

Index(['restaurant_name', 'dining_rating', 'delivery_rating', 'dining_votes',
       'delivery_votes', 'cuisine', 'place_name', 'city', 'item_name',
       'best_seller', 'item_votes', 'item_price'],
      dtype='object')

### B. Duplicate Handling

#### B1. Remove exact duplicate rows

In [13]:
print("Total Records Before Removing Duplicates = ",zomato.duplicated().sum())

Total Records Before Removing Duplicates =  22127


In [16]:
zomato = zomato.drop_duplicates()

In [17]:
print("Total Records After Removing Duplicates = ",zomato.duplicated().sum())

Total Records After Removing Duplicates =  0


### C. Missing Value Framework

#### C1. Best_Seller

In [18]:
zomato['best_seller'].unique()

array(['BESTSELLER', nan, 'MUST TRY', "CHEF'S SPECIAL", 'SEASONAL',
       'Not eligible for coupons', 'VEGAN', 'Not on Pro', 'SPICY', 'NEW',
       'GLUTEN FREE', 'DAIRY FREE', 'FODMAP FRIENDLY',
       'Eggless available'], dtype=object)

In [19]:
# 1. Define the real bestseller taxonomy

bestseller_tags = {
    'BESTSELLER', 'MUST TRY', "CHEF'S SPECIAL",
    'SEASONAL', 'SPICY', 'NEW'
}

dietary_tags = {
    'VEGAN', 'GLUTEN FREE', 'DAIRY FREE',
    'FODMAP FRIENDLY', 'Eggless available'
}

platform_tags = {
    'Not eligible for coupons', 'Not on Pro'
}

In [20]:
# 2. Clean, standardized bestseller category
zomato['best_seller_category'] = zomato['best_seller'].apply(
    lambda x: x if x in bestseller_tags else 'NOT MARKED'
)

In [21]:
# 3. Missingness/marking flag — true only when a REAL bestseller tag exists
zomato['is_best_seller_marked'] = zomato['best_seller'].isin(bestseller_tags).astype(int)

In [22]:
# 4. Preserve the leaked info instead of discarding it
zomato['dietary_tag'] = zomato['best_seller'].apply(
    lambda x: x if x in dietary_tags else np.nan
)
zomato['platform_flag'] = zomato['best_seller'].apply(
    lambda x: x if x in platform_tags else np.nan
)

In [23]:
print(zomato['best_seller_category'].value_counts())
print(zomato['is_best_seller_marked'].value_counts())

best_seller_category
NOT MARKED        84758
BESTSELLER         9884
MUST TRY           4106
CHEF'S SPECIAL     1332
SPICY               993
NEW                 375
SEASONAL             82
Name: count, dtype: int64
is_best_seller_marked
0    84758
1    16772
Name: count, dtype: int64


#### C2. Dining Rating

In [24]:
zomato['dining_rating'].unique()

array([3.9, 4.3, 3.6, 4.2, 4.4, 4.1, 4. , nan, 3.4, 3.2, 3.8, 3.1, 2.7,
       3.7, 3.5, 3. , 2.9, 3.3, 4.6, 4.5, 2.8, 4.7, 2.5, 2.6, 4.8])

In [28]:
zomato['is_dining_rating_missing'] = zomato['dining_rating'].isna().astype(int)

In [29]:
print(f"Missing rate: {zomato['is_dining_rating_missing'].mean():.2%}")

Missing rate: 26.55%


In [30]:
### - BI Dashboard

# Average rating — nulls naturally excluded by pandas
avg_rating_by_city = zomato.groupby('city')['dining_rating'].mean()

# Missingness shown separately, not baked into the average
missingness_by_city = zomato.groupby('city')['is_dining_rating_missing'].mean()

In [31]:
### - Modelling

group_median = zomato.groupby(['city', 'cuisine'])['dining_rating'].transform('median')
city_median = zomato.groupby('city')['dining_rating'].transform('median')
global_median = zomato['dining_rating'].median()

zomato['dining_rating_imputed'] = (
    zomato['dining_rating']
    .fillna(group_median)
    .fillna(city_median)
    .fillna(global_median)
)

# sanity checks
print(zomato['dining_rating_imputed'].isna().sum(), "remaining nulls after imputation")
print(zomato[['dining_rating', 'dining_rating_imputed', 'is_dining_rating_missing']].sample(10))

0 remaining nulls after imputation
       dining_rating  dining_rating_imputed  is_dining_rating_missing
20031            4.2                    4.2                         0
22952            4.2                    4.2                         0
37263            4.7                    4.7                         0
40821            3.8                    3.8                         0
78442            2.7                    2.7                         0
77697            3.6                    3.6                         0
97013            4.3                    4.3                         0
36965            3.8                    3.8                         0
29546            NaN                    3.5                         1
8447             4.2                    4.2                         0


#### C3. Delivery Rating

In [32]:
zomato['is_delivery_rating_missing'] = zomato['delivery_rating'].isna().astype(int)
print(f"Missing rate: {zomato['is_delivery_rating_missing'].mean():.2%}")

Missing rate: 1.23%


In [34]:
# 2. Imputation hierarchy: restaurant -> city+cuisine -> city -> global
restaurant_median = zomato.groupby('restaurant_name')['delivery_rating'].transform('median')
citycuisine_median = zomato.groupby(['city', 'cuisine'])['delivery_rating'].transform('median')
city_median = zomato.groupby('city')['delivery_rating'].transform('median')
global_median = zomato['delivery_rating'].median()

zomato['delivery_rating_imputed'] = (
    zomato['delivery_rating']
    .fillna(restaurant_median)
    .fillna(citycuisine_median)
    .fillna(city_median)
    .fillna(global_median)
)

In [35]:
print(zomato['delivery_rating_imputed'].isna().sum(), "remaining nulls after imputation")
print(zomato[['delivery_rating', 'delivery_rating_imputed', 'is_delivery_rating_missing']].sample(10))

0 remaining nulls after imputation
        delivery_rating  delivery_rating_imputed  is_delivery_rating_missing
95020               4.1                      4.1                           0
99666               3.9                      3.9                           0
121443              3.7                      3.7                           0
21186               4.1                      4.1                           0
4284                3.9                      3.9                           0
32652               4.0                      4.0                           0
80887               4.1                      4.1                           0
23913               4.3                      4.3                           0
81173               3.6                      3.6                           0
26993               4.2                      4.2                           0


In [36]:
zomato['fulfillment_score'] = (
    zomato['delivery_rating_imputed'] * 0.6 +
    zomato['dining_rating_imputed'].fillna(zomato['delivery_rating_imputed']) * 0.4
)

In [37]:
avg_delivery_rating_by_city = zomato.groupby('city')['delivery_rating'].mean()
missingness_by_cuisine = zomato.groupby('cuisine')['is_delivery_rating_missing'].mean()

#### C4. Votes Fields

In [38]:
votes_cols = ['dining_votes', 'delivery_votes', 'item_votes']

for col in votes_cols:
    zomato[col] = pd.to_numeric(
        zomato[col].astype(str).str.replace(',', '').str.strip(),
        errors='coerce'
    )

    neg_count = (zomato[col] < 0).sum()
    if neg_count > 0:
        print(f"{col}: {neg_count} negative values found — clipping to NaN for review")
        zomato.loc[zomato[col] < 0, col] = np.nan

    zomato[f'is_{col}_missing'] = zomato[col].isna().astype(int)

    zomato[col] = zomato[col].fillna(0)

    zomato[col] = zomato[col].astype(int)

for col in votes_cols:
    print(col, "-> min:", zomato[col].min(), "| zeros:", (zomato[col] == 0).sum(),
          "| missing flagged:", zomato[f'is_{col}_missing'].sum())

dining_votes -> min: 0 | zeros: 35080 | missing flagged: 0
delivery_votes -> min: 0 | zeros: 73963 | missing flagged: 0
item_votes -> min: 0 | zeros: 63000 | missing flagged: 0


### D. Type Standardization

#### D1. Numeric Fields

In [39]:
zomato.info()

<class 'pandas.core.frame.DataFrame'>
Index: 101530 entries, 0 to 123635
Data columns (total 24 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   restaurant_name             101530 non-null  object 
 1   dining_rating               74572 non-null   float64
 2   delivery_rating             100286 non-null  float64
 3   dining_votes                101530 non-null  int32  
 4   delivery_votes              101530 non-null  int32  
 5   cuisine                     101530 non-null  object 
 6   place_name                  101530 non-null  object 
 7   city                        101530 non-null  object 
 8   item_name                   101530 non-null  object 
 9   best_seller                 19143 non-null   object 
 10  item_votes                  101530 non-null  int32  
 11  item_price                  101530 non-null  float64
 12  best_seller_category        101530 non-null  object 
 13  is_best_seller_mark

### E. Text Standardization

#### E1. Trim Extra Spaces

In [47]:
string_cols = [
    'restaurant_name', 'cuisine', 'place_name',
    'city', 'item_name', 'best_seller'
]

for col in string_cols:
    if col not in zomato.columns:
        print(f"WARNING: '{col}' not found in columns — check exact name")
        continue

    zomato[col] = zomato[col].astype(str).str.strip()

    zomato[col] = zomato[col].str.replace(r'\s+', ' ', regex=True)

    zomato[col] = zomato[col].replace({'nan': np.nan, 'None': np.nan, '': np.nan})

for col in string_cols:
    if col in zomato.columns:
        bad = zomato[col].dropna().apply(lambda x: x != x.strip() or '  ' in x)
        print(f"{col}: {bad.sum()} rows still with spacing issues")

restaurant_name: 0 rows still with spacing issues
cuisine: 0 rows still with spacing issues
place_name: 0 rows still with spacing issues
city: 0 rows still with spacing issues
item_name: 0 rows still with spacing issues
best_seller: 0 rows still with spacing issues


#### E2. Normalize Casing

In [48]:
import re

casing_cols = ['restaurant_name', 'cuisine', 'place_name', 'city', 'item_name']

brand_overrides = {
    "mcdonald's": "McDonald's",
    "kfc": "KFC",
    "kfc's": "KFC's",
    "burger king": "Burger King",
    "kfc india": "KFC India",
    "domino's": "Domino's",
    "domino's pizza": "Domino's Pizza",
    "kfc's": "KFC's",
    "kfc.": "KFC.",
    "kfc & co": "KFC & Co",
    "kfc-express": "KFC-Express",
}

def smart_title_case(text):
    if pd.isna(text):
        return text

    text_stripped = text.strip()
    lower = text_stripped.lower()

    if lower in brand_overrides:
        return brand_overrides[lower]

    titled = text_stripped.title()

    titled = re.sub(r"'S\b", "'s", titled)

    titled = re.sub(r"\bMc(\w)", lambda m: "Mc" + m.group(1).upper(), titled)
    titled = re.sub(r"\bO'(\w)", lambda m: "O'" + m.group(1).upper(), titled)

    return titled

for col in casing_cols:
    if col not in zomato.columns:
        print(f"WARNING: '{col}' not found — check exact name")
        continue
    zomato[col] = zomato[col].apply(smart_title_case)

for col in casing_cols:
    if col in zomato.columns:
        print(col, "->", zomato[col].dropna().unique()[:15])

restaurant_name -> ['Doner King' 'Taco Bell' 'Brownbear' 'Crystal Restaurant & Bar'
 'Siddique Kabab Centre' 'Shah Ghouse Special Shawarma'
 'Taj Mahal - Taj Mahal Hotel' 'Burger King' 'Papadams Blue'
 'Ovenstory Pizza' 'Italian Pizzeria' 'Al Rabea Mandi House' 'Govind Dosa'
 'Universal Al Mataam Mandi Kebab & Biryani' 'The Thickshake Factory']
cuisine -> ['Fast Food' 'Wraps' 'Biryani' 'Chinese' 'Beverages' 'Desserts' 'Shake'
 'Mandi' 'South Indian' 'Kebab' 'Pizza' 'Bakery' 'Ice Cream' 'Mughlai'
 'North Indian']
place_name -> ['Malakpet' 'The Next Galleria Mall' 'Himayath Nagar' 'Nallakunta'
 'Tolichowki' 'Charminar' 'Taj Mahal Hotel' 'Kothapet' 'Abids'
 'Mehdipatnam' 'Yousufguda' 'Dilsukhnagar' 'Koti' 'Falaknuma'
 'Begum Bazaar']
city -> ['Hyderabad' 'Mumbai' 'Chennai' 'Pune' 'Jaipur' 'Kochi' 'Goa' 'Bangalore'
 'Kolkata' 'Ahmedabad' 'Banaswadi' 'Ulsoor' 'Malleshwaram' 'Magrath Road'
 'Lucknow']
item_name -> ['Platter Kebab Combo' 'Chicken Rumali Shawarma' 'Chicken Tandoori Salad'
 'Ch

#### E3. Remove Hidden Formatting Noise

In [49]:
import re
import unicodedata

text_cols = [
    'restaurant_name', 'cuisine', 'place_name',
    'city', 'item_name', 'best_seller'
]

def clean_hidden_noise(text):
    if pd.isna(text):
        return text

    s = str(text)

    s = unicodedata.normalize('NFKC', s)

    s = s.replace('\xa0', ' ')
    s = s.replace('\u200b', '')
    s = s.replace('\t', ' ').replace('\n', ' ').replace('\r', ' ')

    s = re.sub(r'\s*[;|/]\s*', ', ', s)

    s = re.sub(r',\s*,+', ',', s)

    s = re.sub(r"[^\w\s',&\-.]", '', s)

    s = re.sub(r'\s+', ' ', s).strip()

    if s.lower() in ('', 'nan', 'none'):
        return np.nan

    return s

for col in text_cols:
    if col not in zomato.columns:
        print(f"WARNING: '{col}' not found — check exact name")
        continue
    zomato[col] = zomato[col].apply(clean_hidden_noise)

for col in text_cols:
    if col in zomato.columns:
        print(col, "->", zomato[col].dropna().unique()[:15])

restaurant_name -> ['Doner King' 'Taco Bell' 'Brownbear' 'Crystal Restaurant & Bar'
 'Siddique Kabab Centre' 'Shah Ghouse Special Shawarma'
 'Taj Mahal - Taj Mahal Hotel' 'Burger King' 'Papadams Blue'
 'Ovenstory Pizza' 'Italian Pizzeria' 'Al Rabea Mandi House' 'Govind Dosa'
 'Universal Al Mataam Mandi Kebab & Biryani' 'The Thickshake Factory']
cuisine -> ['Fast Food' 'Wraps' 'Biryani' 'Chinese' 'Beverages' 'Desserts' 'Shake'
 'Mandi' 'South Indian' 'Kebab' 'Pizza' 'Bakery' 'Ice Cream' 'Mughlai'
 'North Indian']
place_name -> ['Malakpet' 'The Next Galleria Mall' 'Himayath Nagar' 'Nallakunta'
 'Tolichowki' 'Charminar' 'Taj Mahal Hotel' 'Kothapet' 'Abids'
 'Mehdipatnam' 'Yousufguda' 'Dilsukhnagar' 'Koti' 'Falaknuma'
 'Begum Bazaar']
city -> ['Hyderabad' 'Mumbai' 'Chennai' 'Pune' 'Jaipur' 'Kochi' 'Goa' 'Bangalore'
 'Kolkata' 'Ahmedabad' 'Banaswadi' 'Ulsoor' 'Malleshwaram' 'Magrath Road'
 'Lucknow']
item_name -> ['Platter Kebab Combo' 'Chicken Rumali Shawarma' 'Chicken Tandoori Salad'
 'Ch

### F. Categorical Standardization

In [50]:
import re
from difflib import get_close_matches

# F1. Cuisine — standardize spelling/labels
zomato['cuisine'] = zomato['cuisine'].str.strip()

cuisine_map = {
    'north indian': 'North Indian',
    'south indian': 'South Indian',
    'fast food': 'Fast Food',
    'fastfood': 'Fast Food',
    'chinese': 'Chinese',
    'chineese': 'Chinese',
    'continental': 'Continental',
    'mughlai': 'Mughlai',
    'mughalai': 'Mughlai',
    'italian': 'Italian',
    'biryani': 'Biryani',
    'desserts': 'Desserts',
    'dessert': 'Desserts',
    'bakery': 'Bakery',
    'beverages': 'Beverages',
    'beverage': 'Beverages',
    'street food': 'Street Food',
    'streetfood': 'Street Food',
}

def standardize_cuisine(val):
    if pd.isna(val):
        return val
    key = val.strip().lower()
    if key in cuisine_map:
        return cuisine_map[key]
    match = get_close_matches(key, cuisine_map.keys(), n=1, cutoff=0.85)
    if match:
        return cuisine_map[match[0]]
    return val.strip().title()

zomato['cuisine'] = zomato['cuisine'].apply(standardize_cuisine)

print("Cuisine unique count:", zomato['cuisine'].nunique())
print(zomato['cuisine'].value_counts().head(20))

Cuisine unique count: 48
cuisine
Beverages       32818
Pizza           12383
Desserts         9285
Fast Food        9268
Chinese          5066
Sichuan          4812
Biryani          3462
Shake            2784
North Indian     2436
Street Food      2371
Rolls            1565
South Indian     1512
Seafood          1284
Ice Cream         966
Mughlai           951
Bakery            934
Momos             856
Bbq               699
Kebab             665
Burger            579
Name: count, dtype: int64


In [51]:
# F2. City — standardize + flag locality-like entries
zomato['city'] = zomato['city'].str.strip().str.title()

city_map = {
    'Bengaluru': 'Bangalore',
    'Bombay': 'Mumbai',
    'Gurugram': 'Gurgaon',
    'New Delhi': 'Delhi',
    'Ncr': 'Delhi NCR',
}
zomato['city'] = zomato['city'].replace(city_map)

known_cities = {
    'Mumbai', 'Delhi', 'Bangalore', 'Hyderabad', 'Chennai', 'Kolkata',
    'Pune', 'Ahmedabad', 'Jaipur', 'Lucknow', 'Chandigarh', 'Kochi',
    'Delhi NCR', 'Gurgaon', 'Noida', 'Indore', 'Bhopal'
}

zomato['city_looks_like_locality'] = ~zomato['city'].isin(known_cities)

suspect = zomato.loc[zomato['city_looks_like_locality'], 'city'].value_counts()
print("Entries in 'city' that may actually be localities, not cities:")
print(suspect.head(30))

Entries in 'city' that may actually be localities, not cities:
city
Raipur          6272
Goa             2287
Banaswadi         85
Ulsoor            59
Magrath Road      45
Malleshwaram      31
Name: count, dtype: int64


In [52]:
# F3. Place_Name -> rename to 'locality'
zomato = zomato.rename(columns={'place_name': 'locality'})
zomato['locality'] = zomato['locality'].str.strip().str.title()

print("Sample localities:", zomato['locality'].dropna().unique()[:20])

Sample localities: ['Malakpet' 'The Next Galleria Mall' 'Himayath Nagar' 'Nallakunta'
 'Tolichowki' 'Charminar' 'Taj Mahal Hotel' 'Kothapet' 'Abids'
 'Mehdipatnam' 'Yousufguda' 'Dilsukhnagar' 'Koti' 'Falaknuma'
 'Begum Bazaar' 'Chandrayanagutta' 'Mpm Mall' 'Rtc X Roads' 'Saroor Nagar'
 'Ghansi Bazaar']


In [53]:
# F4. Best_Seller — final controlled category list
final_best_seller_map = {
    'bestseller': 'BESTSELLER',
    'must try': 'MUST TRY',
    "chef's special": "CHEF'S SPECIAL",
    'spicy': 'SPICY',
    'new': 'NEW',
    'seasonal': 'SEASONAL',
    'not on pro': 'NOT ON PRO',
    'not eligible for coupons': 'NOT ELIGIBLE FOR COUPONS',
    'eggless available': 'EGGLESS AVAILABLE',
}

unmapped_dietary = {'vegan', 'gluten free', 'dairy free', 'fodmap friendly'}

def standardize_best_seller(val):
    if pd.isna(val):
        return 'NOT MARKED'
    key = str(val).strip().lower()
    if key in final_best_seller_map:
        return final_best_seller_map[key]
    if key in unmapped_dietary:
        return 'NOT MARKED'
    return 'NOT MARKED'

zomato['best_seller_category'] = zomato['best_seller'].apply(standardize_best_seller)

zomato['is_best_seller_marked'] = (
    zomato['best_seller_category'] != 'NOT MARKED'
).astype(int)

print(zomato['best_seller_category'].value_counts())

best_seller_category
NOT MARKED                  82423
BESTSELLER                   9884
MUST TRY                     4106
NOT ELIGIBLE FOR COUPONS     1743
CHEF'S SPECIAL               1332
SPICY                         993
NOT ON PRO                    564
NEW                           375
SEASONAL                       82
EGGLESS AVAILABLE              28
Name: count, dtype: int64


In [54]:
zomato.to_csv("zomato_bi_dataset.csv",index=False)

### 5. Outlier and Anomaly Handling

In [55]:
# A. Price outliers
Q1 = zomato['item_price'].quantile(0.25)
Q3 = zomato['item_price'].quantile(0.75)
IQR = Q3 - Q1
iqr_lower = Q1 - 1.5 * IQR
iqr_upper = Q3 + 1.5 * IQR

p01 = zomato['item_price'].quantile(0.01)
p99 = zomato['item_price'].quantile(0.99)

zomato['item_price_capped'] = zomato['item_price'].clip(lower=p01, upper=p99)

zomato['is_price_outlier'] = (
    (zomato['item_price'] < iqr_lower) | (zomato['item_price'] > iqr_upper)
).astype(int)

print(f"IQR bounds: [{iqr_lower:.2f}, {iqr_upper:.2f}]")
print(f"1st/99th percentile caps: [{p01:.2f}, {p99:.2f}]")
print(f"Flagged price outliers: {zomato['is_price_outlier'].sum()} "
      f"({zomato['is_price_outlier'].mean():.2%})")

print(zomato.loc[zomato['is_price_outlier'] == 1, ['restaurant_name', 'item_name', 'item_price']]
      .sort_values('item_price', ascending=False).head(20))

IQR bounds: [-123.50, 552.50]
1st/99th percentile caps: [23.79, 959.00]
Flagged price outliers: 4621 (4.55%)
                    restaurant_name  \
72760                       Arsalan   
31033              Khalids Biriyani   
100934         The Hazelnut Factory   
31102              Khalids Biriyani   
31101              Khalids Biriyani   
100933         The Hazelnut Factory   
92334               Behrouz Biryani   
111884              Behrouz Biryani   
122551              Behrouz Biryani   
25059                   Boojee Cafe   
70188            Ambur Star Briyani   
111887              Behrouz Biryani   
122554              Behrouz Biryani   
38566   Kms Hakkim Kalyana Biriyani   
38565   Kms Hakkim Kalyana Biriyani   
94434                Bucket Biryani   
31042              Khalids Biriyani   
23827                  Tossin Pizza   
100931         The Hazelnut Factory   
100932         The Hazelnut Factory   

                                                item_name  item_price  

In [56]:
# B. Vote outliers

vote_cols = ['dining_votes', 'delivery_votes', 'item_votes']

for col in vote_cols:
    p99 = zomato[col].quantile(0.99)
    zomato[f'{col}_capped'] = zomato[col].clip(upper=p99)

    Q1 = zomato[col].quantile(0.25)
    Q3 = zomato[col].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.5 * IQR

    zomato[f'is_{col}_outlier'] = (zomato[col] > upper_bound).astype(int)

    print(f"{col}: 99th pct cap = {p99:.0f}, IQR upper bound = {upper_bound:.0f}, "
          f"flagged = {zomato[f'is_{col}_outlier'].sum()} "
          f"({zomato[f'is_{col}_outlier'].mean():.2%})")

print(zomato.loc[zomato['is_item_votes_outlier'] == 1,
                  ['restaurant_name', 'item_name', 'item_votes']]
      .sort_values('item_votes', ascending=False).head(20))

dining_votes: 99th pct cap = 901, IQR upper bound = 552, flagged = 8376 (8.25%)
delivery_votes: 99th pct cap = 944, IQR upper bound = 92, flagged = 23513 (23.16%)
item_votes: 99th pct cap = 292, IQR upper bound = 25, flagged = 13766 (13.56%)
                       restaurant_name                item_name  item_votes
12417                           Mehfil   Chicken Biryani Single        9750
11146                         Bawarchi     Mini Chicken Biryani        8613
6135                  Lucky Restaurant          Chicken Biryani        7931
6131                  Lucky Restaurant          Chicken Biryani        7931
11145                         Bawarchi          Chicken Biryani        7586
12416                           Mehfil     Chicken Biryani Full        6753
109977                  Lazeez Biryani          Chicken Biryani        4459
110006                  Lazeez Biryani          Chicken Biryani        4459
13305      Al Rabea Al Arabi Cafeteria             Chicken Wrap        414

In [57]:
zomato.to_csv("zomato_recommendation_dataset.csv",index=False)